## Setting up

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # training/Unsloth uses GPU 0 only

In [4]:
!pip install -q unsloth
# Also get the latest nightly Unsloth!
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.4/30.4 MB 71.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 89.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 14.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Pre

In [5]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! Unsloth also supports RoPE (Rotary Positinal Embedding) scaling internally.
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, # Will load the 4Bit Quantized Model
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.6: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, # a higher alpha value assigns more weight to the LoRA activations
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.10.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


**Load the data**

In [7]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/forouzanrezaei/sentiment-analysis-for-mental-health/Combined Data.csv",index_col = "Unnamed: 0")
df.loc[:,'status'] = df.loc[:,'status'].str.replace('Bi-Polar','Bipolar')
df = df[(df.status != "Personality disorder") & (df.status != "Stress") & (df.status != "Suicidal")]
df.head()

,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38520 entries, 0 to 53042
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   statement  38365 non-null  object
 1   status     38520 non-null  object
dtypes: object(2)
memory usage: 902.8+ KB


In [9]:
df.isnull().sum()

statement    155
status         0
dtype: int64

In [10]:
df = df.dropna(subset=['statement'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38365 entries, 0 to 53042
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   statement  38365 non-null  object
 1   status     38365 non-null  object
dtypes: object(2)
memory usage: 899.2+ KB


In [11]:
df.isnull().sum()

statement    0
status       0
dtype: int64

In [12]:
df = df.reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38365 entries, 0 to 38364
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   statement  38365 non-null  object
 1   status     38365 non-null  object
dtypes: object(2)
memory usage: 599.6+ KB


In [13]:
# Shuffle the DataFrame and select only 3000 rows
df = df.sample(frac=1, random_state=85).reset_index(drop=True).head(3000)

# Split the DataFrame
train_size = 0.8
eval_size = 0.1

# Calculate sizes
train_end = int(train_size * len(df))
eval_end = train_end + int(eval_size * len(df))

# Split the data
X_train = df[:train_end]
X_eval = df[train_end:eval_end]
X_test = df[eval_end:]

# # Define the prompt generation functions
# def generate_prompt(data_point):
#     return f"""
#             Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.
# text: {data_point["statement"]}
# label: {data_point["status"]}""".strip()

EOS = "<|im_end|>"

def generate_prompt(data_point):
    return f"""
            Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.
text: {data_point["statement"]}
label: {data_point["status"]}{EOS}""".strip()

def generate_test_prompt(data_point):
    return f"""
            Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.
text: {data_point["statement"]}
label: """.strip()

# Generate prompts for training and evaluation data
X_train.loc[:,'text'] = X_train.apply(generate_prompt, axis=1)
X_eval.loc[:,'text'] = X_eval.apply(generate_prompt, axis=1)

# Generate test prompts and extract true labels
y_true = X_test.loc[:,'status']
X_test = pd.DataFrame(X_test.apply(generate_test_prompt, axis=1), columns=["text"])



/tmp/ipykernel_48/517379280.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.loc[:,'text'] = X_train.apply(generate_prompt, axis=1)
/tmp/ipykernel_48/517379280.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_eval.loc[:,'text'] = X_eval.apply(generate_prompt, axis=1)


In [14]:
X_train.status.value_counts()

status
Normal        1020
Depression     969
Anxiety        246
Bipolar        165
Name: count, dtype: int64

In [15]:
y_true.value_counts()

status
Depression    130
Normal        116
Anxiety        31
Bipolar        23
Name: count, dtype: int64

In [16]:
from datasets import Dataset
# Convert to datasets
train_data = Dataset.from_pandas(X_train[["text"]])
eval_data = Dataset.from_pandas(X_eval[["text"]])

In [17]:
train_data['text'][3]

'Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.\ntext: I have always wanted to had had urges to but I have never actually tried. I want to so bad but I am stuck here because other people in my family died and my little brother is still just a kid. I am hurting so bad but my brain says only people who try to kill themself or have worse disorders should go. If I try to I really 100% think Ill die and I wish I had an easy moral out so bad. I feel so fake because I have never tried to kill myself.\nlabel: Depression<|im_end|>'

In [18]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Number of processors to use for processing the dataset
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2, # The batch size per GPU/TPU core
        gradient_accumulation_steps = 4, # Number of steps to perform befor each gradient accumulation
        warmup_steps = 5, # Few updates with low learning rate before actual training
        num_train_epochs = 1, # Specifies the total number of training steps (batches) to run.
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc for observability
    ),
)

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2400 [00:00<?, ? examples/s]

In [19]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,400 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,3.425660
2,3.760065
3,3.394748
4,3.711992
5,3.533228
6,2.862779
7,2.802054
8,2.858384
9,2.562171
10,2.191411


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.


In [20]:
%%capture
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

In [21]:
import numpy as np
import transformers
from transformers import pipeline
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


transformers.logging.set_verbosity_error()

In [22]:
def predict(test, model, tokenizer):
    y_pred = []
    categories = ["Normal", "Depression", "Anxiety", "Bipolar"]

    pipe = pipeline(task="text-generation",
                    model=model,
                    tokenizer=tokenizer,
                    max_new_tokens=6,
                    do_sample=False,
                    return_full_text=False)

    for i in tqdm(range(len(test))):
        result = pipe(test.iloc[i]["text"])
        answer = result[0]["generated_text"].strip()

        for category in categories:
            if category.lower() in answer.lower():
                y_pred.append(category)
                break
        else:
            y_pred.append("none")

    return y_pred

In [23]:
y_pred = predict(X_test, model, tokenizer)

  0%|          | 0/300 [00:00<?, ?it/s]

In [24]:
def evaluate(y_true, y_pred):
    labels = ["Normal", "Depression", "Anxiety", "Bipolar"]
    mapping = {label: idx for idx, label in enumerate(labels)}
    
    def map_func(x):
        return mapping.get(x, -1)  # Map to -1 if not found, but should not occur with correct data
    
    y_true_mapped = np.vectorize(map_func)(y_true)
    y_pred_mapped = np.vectorize(map_func)(y_pred)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true_mapped, y_pred=y_pred_mapped)
    print(f'Accuracy: {accuracy:.3f}')
    
    # Generate accuracy report
    unique_labels = set(y_true_mapped)  # Get unique labels
    
    for label in unique_labels:
        label_indices = [i for i in range(len(y_true_mapped)) if y_true_mapped[i] == label]
        label_y_true = [y_true_mapped[i] for i in label_indices]
        label_y_pred = [y_pred_mapped[i] for i in label_indices]
        label_accuracy = accuracy_score(label_y_true, label_y_pred)
        print(f'Accuracy for label {labels[label]}: {label_accuracy:.3f}')
        
    # Generate classification report
    class_report = classification_report(y_true=y_true_mapped, y_pred=y_pred_mapped, target_names=labels, labels=list(range(len(labels))))
    print('\nClassification Report:')
    print(class_report)
    
    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true_mapped, y_pred=y_pred_mapped, labels=list(range(len(labels))))
    print('\nConfusion Matrix:')
    print(conf_matrix)

In [25]:
evaluate(y_true, y_pred)

Accuracy: 0.887
Accuracy for label Normal: 0.905
Accuracy for label Depression: 0.962
Accuracy for label Anxiety: 0.613
Accuracy for label Bipolar: 0.739

Classification Report:
              precision    recall  f1-score   support

      Normal       0.95      0.91      0.93       116
  Depression       0.86      0.96      0.91       130
     Anxiety       0.76      0.61      0.68        31
     Bipolar       0.94      0.74      0.83        23

    accuracy                           0.89       300
   macro avg       0.88      0.80      0.83       300
weighted avg       0.89      0.89      0.88       300


Confusion Matrix:
[[105   8   3   0]
 [  1 125   3   1]
 [  4   8  19   0]
 [  1   5   0  17]]


In [26]:
model.save_pretrained("qwen2.5-1.5b-mental-health")
tokenizer.save_pretrained("qwen2.5-1.5b-mental-health")

('qwen2.5-1.5b-mental-health/tokenizer_config.json',
 'qwen2.5-1.5b-mental-health/chat_template.jinja',
 'qwen2.5-1.5b-mental-health/tokenizer.json')

In [27]:
model.save_pretrained_gguf("qwen2.5-1.5b-mental-health-GGUF", tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:09<00:00,  9.57s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:14<00:00, 14.86s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen2.5-1.5b-mental-health-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11696-mix-7baeb9a (app-b11696-mix-7baeb9a-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen2.5-1.5b-mental-health-GGUF_gguf/Qwen2.5-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...

{'save_directory': 'qwen2.5-1.5b-mental-health-GGUF',
 'gguf_directory': 'qwen2.5-1.5b-mental-health-GGUF_gguf',
 'gguf_files': ['qwen2.5-1.5b-mental-health-GGUF_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'qwen2.5-1.5b-mental-health-GGUF_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [28]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 125 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (348 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 129802 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video gro

In [29]:
# %%capture
# import subprocess
# subprocess.Popen(["ollama", "serve"])
# import time
# time.sleep(3)

# !pkill ollama
# import subprocess, time
# subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# time.sleep(3)

import subprocess, time

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "1"   # Ollama uses GPU 1 only

subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

In [30]:
# # print(tokenizer._ollama_modelfile)
# !ollama create qwen_mental_health -f ./qwen2.5-1.5b-mental-health-GGUF/Modelfile

In [31]:
!ls qwen2.5-1.5b-mental-health-GGUF

chat_template.jinja  generation_config.json  tokenizer_config.json
config.json	     model.safetensors	     tokenizer.json


In [32]:
# modelfile = '''FROM ./qwen2.5-1.5b-mental-health-GGUF
# TEMPLATE """{{ .Prompt }}"""
# PARAMETER stop "<|im_end|>"
# PARAMETER temperature 0
# '''
# with open("Modelfile", "w") as f:
#     f.write(modelfile)

# !ollama create qwen_mental_health -f Modelfile


modelfile = '''FROM ./qwen2.5-1.5b-mental-health-GGUF_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf
TEMPLATE """{{ .Prompt }}"""
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0
'''
with open("Modelfile", "w") as f:
    f.write(modelfile)

!ollama create qwen_mental_health -f Modelfile

]11;?\gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠋ gathering model components ⠹ gathering model components ⠸ gathering model components ⠸ gathering model components ⠴ gathering model components ⠴ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠏ gathering model components ⠋ gathering model components ⠹ gathering model components ⠸ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠧ gathering model components ⠧ gathering model components ⠏ gathering model components ⠋ gathering model components 
copying file sha256:529accd9b91f2ebe6ae835cef89e024f75e0b007dc15b607772c474fe5d1906f 1% ⠋ gath

In [33]:
# %%capture
# !git clone --depth 1 https://github.com/ggml-org/llama.cpp
# !pip install -q gguf sentencepiece
# !python llama.cpp/convert_hf_to_gguf.py qwen2.5-1.5b-mental-health-GGUF --outfile qwen-mental-health.gguf --outtype q8_0



In [34]:
# %%capture
# modelfile = '''FROM ./qwen-mental-health.gguf
# TEMPLATE """{{ .Prompt }}"""
# PARAMETER stop "<|im_end|>"
# PARAMETER temperature 0
# '''
# with open("Modelfile", "w") as f:
#     f.write(modelfile)

# !ollama create qwen_mental_health -f Modelfile

In [35]:
# %%capture
# !ollama create qwen_mental_health -f ./qwen2.5-1.5b-mental-health-GGUF_gguf/Modelfile

In [36]:
# !ls -lh qwen-mental-health.gguf


In [37]:
# %%capture
# import subprocess

# prompt = """Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.
# text: I can't stop worrying about everything
# label:"""

# result = subprocess.run(["ollama", "run", "qwen_mental_health", prompt],
#                         capture_output=True, text=True)
# print(result.stdout.strip())

import subprocess

prompt = """Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.
text: I can't stop worrying about everything
label:"""

result = subprocess.run(["ollama", "run", "qwen_mental_health", prompt],
                        capture_output=True, text=True)
print(result.stdout.strip())

Depression


In [38]:
# y_pred = predict(X_test, model, tokenizer)
# evaluate(y_true, y_pred)